EL_Process

In [ ]:
import os
import shutil
import subprocess

container = "hadoop-docker-itvdelab-1"
local_src = "/data/Raw_Looker_Ecommerce_BigQuery_Dataset"
bronze = "/Data/Bronze"
expected_files = ["users.csv", "products.csv", "distribution_centers.csv", "order_items.csv", "events.csv"]

def hdfs(args):
    
    cmd = f"hdfs dfs {args}" if shutil.which("hdfs") else f"docker exec {container} hdfs dfs {args}"
    r = subprocess.run(cmd, shell=True,
                       stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                       universal_newlines=True)
    if r.returncode != 0:
        raise RuntimeError(f"hdfs dfs {args} failed:\n{r.stderr}")
    return r.stdout

dataset_dir = os.path.basename(local_src.rstrip("/"))   
bronze_data = f"{bronze}/{dataset_dir}"   

hdfs(f"-mkdir -p {bronze}")
hdfs(f"-put -f {local_src}/ {bronze}/")  

listing = hdfs(f"-ls {bronze_data}")
missing = [f for f in expected_files if f not in listing]
if missing:
    raise RuntimeError(f"Bronze upload incomplete, missing: {missing}")
print(listing)

Analyze Looker Ecommerce BigQuery Dataset

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark=SparkSession.builder \
                  .appName('bronze_layer') \
                  .master('local[*]') \
                  .config('spark.sql.shuffle.partitions', '8') \
                  .config('spark.driver.memory', '2g') \
                  .getOrCreate()

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

def check_nulls(df):
    print("Null values in each column:")
    df.select([
        F.sum(col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ]).show()

def check_duplicates(df, column):
    print(f"Duplicate values in {column}:")
    df.groupBy(column).count() \
      .filter(col("count") > 1) \
      .show()

def check_distinct(df, columns):
    if isinstance(columns, str):
        columns = [columns]

    for column in columns:
        print(f"Distinct values in {column}:")
        df.select(column).distinct().show(truncate=False)

def check_invalid_values(df, columns, minimum=0, maximum=100000):
    if isinstance(columns, str):
        columns = [columns]

    for column in columns:
        print(f"Invalid values in {column}:")
        df.filter(
            (col(column) <= minimum) |
            (col(column) > maximum)
        ).show()

def check_date_order(df, earlier, later):
    """Rows where `later` happens before `earlier`"""
    bad = df.filter(
        col(earlier).isNotNull() & col(later).isNotNull() &
        (col(later).cast("timestamp") < col(earlier).cast("timestamp"))
    )
    print(f"Rows where {later} is before {earlier}: {bad.count()}")
    bad.show(5, truncate=False)

def check_future_dates(df, column):
    """Timestamps later than now"""
    n = df.filter(col(column).cast("timestamp") > F.current_timestamp()).count()
    print(f"Rows where {column} is in the future: {n}")

def check_orphans(child, child_col, parent, parent_col):
    """Distinct values in child.child_col that have no match in parent.parent_col"""
    c = child.select(col(child_col).cast("long").alias("k")).where(col("k").isNotNull()).distinct()
    p = parent.select(col(parent_col).cast("long").alias("k")).distinct()
    print(f"Values of {child_col} with no matching {parent_col}: {c.join(p, on='k', how='left_anti').count()}")

users_dataframe

In [ ]:
users_df = spark.read.csv(f"hdfs://{bronze_data}/users.csv",
                          header=True,
                          inferSchema=True,
                          nullValue='null')

In [ ]:
users_df.printSchema()

In [ ]:
check_nulls(users_df)

check_duplicates(users_df, 'id')

check_distinct(users_df, ['gender','traffic_source'])

check_invalid_values(users_df, 'age', minimum=0, maximum=100)

check_future_dates(users_df, 'created_at')

products_dataframe

In [ ]:
products_df = spark.read.csv(f"hdfs://{bronze_data}/products.csv",
                          header=True,
                          inferSchema=True,
                          nullValue='null')

In [ ]:
products_df.printSchema()

In [ ]:
check_nulls(products_df)

check_duplicates(products_df, 'id')

check_distinct(products_df, ['category','department'])

check_invalid_values(products_df, ['cost','retail_price'], minimum=0, maximum=100000)

distribution_centers_dataframe

In [ ]:
distribution_centers_df = spark.read.csv(f"hdfs://{bronze_data}/distribution_centers.csv",
                          header=True,
                          inferSchema=True,
                          nullValue='null')

In [ ]:
distribution_centers_df.printSchema()

In [ ]:
check_orphans(products_df, 'distribution_center_id', distribution_centers_df, 'id')

order_items_dataframe

In [ ]:
order_items_df = spark.read.csv(f"hdfs://{bronze_data}/order_items.csv",
                          header=True,
                          inferSchema=True,
                          nullValue='null')

In [ ]:
order_items_df.printSchema()

In [ ]:
check_nulls(order_items_df)

check_duplicates(order_items_df, 'id')

check_distinct(order_items_df, 'status')

check_invalid_values(order_items_df,'sale_price', minimum=0, maximum=100000)

check_date_order(order_items_df, 'created_at', 'shipped_at')

check_date_order(order_items_df, 'shipped_at', 'delivered_at')

check_date_order(order_items_df, 'delivered_at', 'returned_at')

check_orphans(order_items_df, 'user_id', users_df, 'id')

check_orphans(order_items_df, 'product_id', products_df, 'id')

events_dataframe

In [ ]:
events_df = spark.read.csv(f"hdfs://{bronze_data}/events.csv",
                          header=True,
                          inferSchema=True,
                          nullValue='null')

In [ ]:
events_df.printSchema()

In [ ]:
check_nulls(events_df)

check_duplicates(events_df, 'id')

check_distinct(events_df,['sequence_number','browser','traffic_source','event_type'])

check_future_dates(events_df, 'created_at')

check_orphans(events_df, 'user_id', users_df, 'id')